# Ask Your Syllabus — Intro to RAG


In this notebook, we build a **Retrieval-Augmented Generation (RAG)** assistant that answers questions using a **fictional** CS 3650 syllabus.

> All course details are invented for teaching. This is not a real course policy document.


## 0. One-time setup

1. Install **Python**, **Jupyter** (VS Code's Jupyter extension works), and **[Ollama](https://ollama.com/download)**.
    - Windows: 
        ```bash 
        irm https://ollama.com/install.ps1 | iex
        ```
    - MacOS/Linux:
        ```bash
        curl -fsSL https://ollama.com/install.sh | sh
        ```
    
2. Open this folder in VS Code. Ensure the notebook and `syllabus.txt` are in the **same folder**.
3. Run the cell below **once** to install Python packages into the notebook's active Python environment. Restart the kernel only if prompted.
4. From a **terminal**, download the two models:

    ```bash
    ollama pull all-minilm
    ollama pull gemma3:1b
    ```

Make sure Ollama is running. If Ollama isn't already serving, launch the Ollama app or run `ollama serve` in a separate terminal.

In [ ]:
%pip install ollama chromadb

## 1. Import the libraries and select models

- **Ollama:** locally runs the embedding model and the LLM.
- **ChromaDB:** our tiny in-memory vector database.
- **Path:** reads the syllabus file.

In [ ]:
from pathlib import Path
import chromadb
import ollama

EMBED_MODEL = "all-minilm"  # Turns text into numerical vectors
CHAT_MODEL = "gemma3:1b"  # Writes answers based on retrieved text

## 2. Closed-book test — ask the AI BEFORE it sees the syllabus

**The experiment:** We'll ask the *same question twice*. First, the LLM answers from its existing training knowledge **without** seeing `syllabus.txt`. Later, RAG retrieves the course policy and supplies it as context.

Our fictional syllabus contains a specific Homework 3 due date that the model has never been told. The closed-book model should admit uncertainty rather than invent a deadline.

In [ ]:
question = "When is Homework 3 due?"

# No document, vector database, or retrieved text is included here.
closed_response = ollama.chat(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "Answer honestly. If you don't know the course deadline, say so. Do not make up dates."},
        {"role": "user", "content": question},
    ],
)
closed_book_answer = closed_response.message.content

print("QUESTION:", question)
print("\nCLOSED-BOOK ANSWER (no syllabus):\n", closed_book_answer)

## 3. Load the syllabus

The knowledge base is just a text file. We don't retrain the AI; we let it **look up** useful facts.

In [ ]:
syllabus_path = Path("syllabus.txt")

if not syllabus_path.exists():
    raise FileNotFoundError(
        "Put syllabus.txt next to this notebook, then run from that folder. "
        f"Current folder: {Path.cwd()}"
    )

syllabus = syllabus_path.read_text(encoding="utf-8")

print(syllabus[:700])

## 4. Chunk — split the syllabus into small pieces

We split on **blank lines**, so every paragraph becomes one chunk. Real systems use more advanced methods, but this is easy to explain and works with our short sample syllabus.

In [ ]:
chunks = [part.strip() for part in syllabus.split("\n\n") if part.strip()]

print(f"Found {len(chunks)} chunks. Here's chunk 3:\n")
print(chunks[2])

## 5. Embed — convert chunks into vectors

An **embedding** is a list of numbers that represents a piece of text. Similar meanings tend to have similar embeddings.

The cell below calls Ollama's small embedding model. The first time can take a moment.

In [ ]:
# We can create embeddings using Ollama.
chunk_vectors = ollama.embed(model=EMBED_MODEL, input=chunks).embeddings

print("Number of chunks embedded:", len(chunk_vectors))
print("Numbers in each vector:", len(chunk_vectors[0]))
print("First five numbers:", list(chunk_vectors[0])[:5])

## 6. Store — build our vector database

Chroma stores **both** each vector **and** its original text, so we can search using numbers but show the LLM real syllabus words.

This is an **in-memory** database; it resets when you restart the notebook. Re-running this cell also creates a clean database.

In [ ]:
# Create the Chroma DB object
db = chromadb.EphemeralClient()
collection = db.get_or_create_collection(name="syllabus_notebook_demo")

# Clear any old entries if we run this cell more than once.
old_ids = collection.get()["ids"]
if old_ids:
    collection.delete(ids=old_ids)

collection.add(
    ids=[str(i) for i in range(len(chunks))],
    documents=chunks,
    embeddings=[list(vector) for vector in chunk_vectors],
)
print("Stored", collection.count(), "chunks in ChromaDB!")

## 7. Ask a question

Try changing the question, then run **this cell and the two cells below** again.

In [ ]:
# Reuse the same question from our closed-book test.
print("Student asks:", question)


## 8. Retrieve — find the closest syllabus paragraphs

We use the **same embedding model** on the question. Chroma finds the two chunks whose embeddings best match the question.

**Notice:** No LLM answer yet! This is only **retrieval**.

In [ ]:
# Turn the student's question into an embedding
question_vector = ollama.embed(
    model=EMBED_MODEL,
    input=question
).embeddings[0]

# Search for the 5 most similar chunks
matches = collection.query(
    query_embeddings=[list(question_vector)],
    n_results=min(4, collection.count()),
    # The min represents the amount of similar paragraphs it should return
)

# Extract the original text
retrieved_chunks = matches["documents"][0]

# Combine retrieved chunks into context
context = "\n\n".join(retrieved_chunks)

# Display retrieved information
print("RETRIEVED SYLLABUS TEXT:\n")

for i, paragraph in enumerate(retrieved_chunks, start=1):
    print(f"Match {i}: {paragraph}\n")

## 9. Generate — give the retrieved text to the LLM

This is the **augmented generation** part. Instead of expecting the LLM to know our fictional syllabus, we give it **the student's question and retrieved paragraphs**.

RAG can reduce guessing, but it **does not guarantee accuracy**. Always check the retrieved text.

In [ ]:
prompt = f"""Answer using ONLY these syllabus excerpts.
If the answer is not included, say: "I couldn't find that in the syllabus."
Keep the response short. Do not invent details.

SYLLABUS EXCERPTS:
{context}

STUDENT QUESTION:
{question}
"""

response = ollama.chat(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": prompt}],
)

print("AI ANSWER:\n")
print(response.message.content)

## 10. Compare: closed-book vs. open-book

We asked exactly the same question both times. The **closed-book** answer had no course data; the **open-book/RAG** answer received relevant syllabus paragraphs through retrieval.

Notice whether the answer is supported by the retrieved text. RAG is a method for *grounding* answers, not a guarantee of correctness.


In [ ]:
print("QUESTION:", question)
print("\nBEFORE RAG — closed book:\n", closed_book_answer)
print("\nAFTER RAG — open book:\n", response.message.content)
print("\nCHECK THE SOURCE — retrieved text:\n", context)


## 11. Ask More Questions

- `When is Homework 3 due?`
- `What can I bring to the midterm?`
- `How much is the final worth?`
- `Can I use a calculator on the final?` (not stated in our sample syllabus)

**Important:** A missing answer doesn't always mean retrieval was bad; the information might never have been in the file.

In [ ]:
def ask_syllabus(question):

    # STEP 1: Convert the question into an embedding
    vector = ollama.embed(
        model=EMBED_MODEL,
        input=question
    ).embeddings[0]

    # STEP 2: Retrieve relevant syllabus paragraphs
    results = collection.query(
        query_embeddings=[list(vector)],
        n_results=min(5, collection.count())
    )

    context = "\n\n".join(results["documents"][0])

    # STEP 3: Create a clear prompt
    system_prompt = """
    You are a helpful university course assistant.

    Answer the student's question using the provided syllabus.

    Rules:
    - Read the syllabus carefully before answering.
    - Give a direct and concise answer.
    - Questions about how much something is "worth"
        refer to its percentage of the course grade.
    - If the answer is present, use it.
    - If the answer is genuinely missing, say:
        "I couldn't find that in the syllabus."
    - Do not invent information.
    """

    user_prompt = f"""
    SYLLABUS:
    {context}

    QUESTION:
    {question}
    """

    # STEP 4: Generate an answer
    response = ollama.chat(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        options={"temperature": 0}
    )

    # STEP 5: Display the results
    print("QUESTION:", question)
    print("\nRETRIEVED TEXT:\n", context)
    print("\nANSWER:\n", response.message.content)

# Here we can ask questions
ask_syllabus("How much is the final exam worth?")

## 12. Live class experiment — update the knowledge base

1. Ask: **"Can I use a calculator during the final exam?"** (The original syllabus doesn't say.)
2. Open `syllabus.txt` and **add a new paragraph**, for example:

   `During the final exam, students may use a basic scientific calculator. Graphing calculators are not allowed.`

3. Save the text file. **Re-run Steps 3–6** to reload, re-embed, and rebuild the database.
4. Run the question again with the bonus `ask_syllabus(...)` function.

**Key lesson:** The application's knowledge changed even though **neither AI model was retrained**.

In [ ]:
# Try this before and after editing syllabus.txt, rebuilding Steps 2–5.
ask_syllabus("Can I use a calculator during the final exam?")